In [3]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from plot_func import error_scatter, interval_score, coverage, performance_dist, table

# Test Case 1 experimental data
# folder = "interp_reg_spatial"

folder = "synthetic_data/case11/pyvale-output/interp_reg_spatial"

metrics = ["total_plus", "total_minus"]
models = [
    "1", "2", "3", "4", "5", "6", "7", "8",
    "9", "10", "11", "12", "13", "14", "15", "16"
]

TOLERANCE=0.1

In [4]:

for metric in metrics:
    
    dfs = []
    for model in models:
                df = pd.read_csv(f"../../{folder}/ablation_results/{metric}_ablation_{model}.csv")
                df["model"] = model
                dfs.append(df)

    df_all = pd.concat(dfs, ignore_index=True)
    
    fig, ax = error_scatter(df_all, TOLERANCE, tag=metric)
    save_path = f"../../{folder}/plots/error_scatter_{metric}_{model}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = interval_score(df_all, models, tag=metric)
    save_path = f"../../{folder}/plots/interval_score_{metric}_{model}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = coverage(df_all, tag=metric)
    save_path = f"../../{folder}/plots/coverage_{metric}_{model}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
/home/wiera/Documents/fullfieldvalmetrics/scripts/plots/plot_func.py:101: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_all.groupby("model")["within_pi"]
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
/home/wiera/Documents/fullfieldvalmetrics/scripts/plots/plot_func.py:101: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_all.groupby("model")["within_pi"]


In [5]:
all_results = []

for metric in metrics:

    for model in models:

        # filepath = (f"../../{folder}/ablation_results/"
        #             f"{metric}_ablation_{model_type_print}.csv"
        # )
        filepath = f"../../{folder}/ablation_summary_{model}.csv"

        df = pd.read_csv(filepath)

        # Store the model information with each result
        df["metric"] = metric
        df["model_type"] = model


        all_results.append(df)


# Combine all summary files
results = pd.concat(all_results, ignore_index=True)
print(results)

               d_type       MAE       RMSE         MAPE  mean_abs_error  \
0            sim_plus  8.389118  10.296982  1743.336337        8.389118   
1           sim_minus  7.876267   9.661439  1901.051846        7.876267   
2     model_form_plus  1.320731   1.567364  6284.989852        1.320731   
3    model_form_minus  1.782602   2.221590    25.969442        1.782602   
4          total_plus  9.028639  11.185810   135.974704        9.028639   
..                ...       ...        ...          ...             ...   
187         sim_minus  7.876267   9.661439  1901.051846        7.876267   
188   model_form_plus  1.342222   1.585342  7226.412418        1.342222   
189  model_form_minus  1.783437   2.222240    25.984486        1.783437   
190        total_plus  9.288542  11.657020   136.716475        9.288542   
191       total_minus  9.635565  11.858247   102.407460        9.635565   

     mean_rel_error  mean_pi_width  mean_pi_error  pi_coverage  \
0         17.433363      49.24164

In [6]:
best_mean_rel_error = (
    results.loc[
        results.groupby("d_type")["mean_rel_error"].idxmin()
    ]
)

print(best_mean_rel_error[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
])

df_to_plot = best_mean_rel_error[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
]

fig, ax = table(df_to_plot)
save_path = f"../../{folder}/plots/best_mean_rel_error.jpg"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close(fig)

              d_type model_type  mean_rel_error  pi_coverage  \
9   model_form_minus          2        0.130573     1.000000   
8    model_form_plus          2       46.981248     1.000000   
7          sim_minus          2        6.104426     0.857143   
6           sim_plus          2        5.905155     0.857143   
11       total_minus          2        0.368081     0.857143   
10        total_plus          2        0.666257     1.000000   

    mean_interval_score  
9              4.832462  
8             13.072516  
7             22.680783  
6             23.960138  
11            25.673851  
10            23.608554  


In [7]:
best_interval_score = (
    results.loc[
        results.groupby("d_type")["mean_interval_score"].idxmin()
    ]
)

print(best_interval_score[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
])

df_to_plot = best_interval_score[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
]

fig, ax = table(df_to_plot)
save_path = f"../../{folder}/plots/best_interval_score.jpg"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close(fig)

              d_type model_type  mean_rel_error  pi_coverage  \
57  model_form_minus         10        0.131386     0.714286   
92   model_form_plus         16       72.264124     0.857143   
7          sim_minus          2        6.104426     0.857143   
6           sim_plus          2        5.905155     0.857143   
11       total_minus          2        0.368081     0.857143   
10        total_plus          2        0.666257     1.000000   

    mean_interval_score  
57             3.670313  
92             7.019448  
7             22.680783  
6             23.960138  
11            25.673851  
10            23.608554  


In [8]:
target_coverage = 0.95

results["coverage_distance"] = (
    results["pi_coverage"] - target_coverage
).abs()

best_coverage = (
    results.loc[
        results.groupby("d_type")["coverage_distance"].idxmin()
    ]
)

print(best_coverage[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
])

df_to_plot = best_coverage[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
]

fig, ax = table(df_to_plot)
save_path = f"../../{folder}/plots/best_coverage.jpg"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close(fig)


             d_type model_type  mean_rel_error  pi_coverage  \
9  model_form_minus          2        0.130573          1.0   
2   model_form_plus          1       62.849899          1.0   
1         sim_minus          1       19.010518          1.0   
0          sim_plus          1       17.433363          1.0   
5       total_minus          1        1.024075          1.0   
4        total_plus          1        1.359747          1.0   

   mean_interval_score  
9             4.832462  
2             9.480223  
1            46.212912  
0            49.241641  
5            56.553459  
4            54.667023  


In [9]:
results["rel_error_rank"] = (
    results.groupby("d_type")["mean_rel_error"]
    .rank(method="min", ascending=True)
)

results["interval_score_rank"] = (
    results.groupby("d_type")["mean_interval_score"]
    .rank(method="min", ascending=True)
)

results["coverage_rank"] = (
    results.groupby("d_type")["coverage_distance"]
    .rank(method="min", ascending=True)
)

results["overall_rank"] = (
    results["rel_error_rank"]
    + results["interval_score_rank"]
    + results["coverage_rank"]
)

In [10]:
best_overall = (
    results.loc[
        results.groupby("d_type")["overall_rank"].idxmin()
    ]
)

print(best_overall[
    [
        "d_type",
        "model_type",
        "mean_rel_error",
        "pi_coverage",
        "mean_interval_score",
        "overall_rank",
    ]
])


df_to_plot = best_overall[
    [
        "d_type",
        "model_type",
        "mean_rel_error",
        "pi_coverage",
        "mean_interval_score",
        "overall_rank",
    ]
]

fig, ax = table(df_to_plot)
save_path = f"../../{folder}/plots/best_overall.jpg"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close(fig)

              d_type model_type  mean_rel_error  pi_coverage  \
9   model_form_minus          2        0.130573          1.0   
2    model_form_plus          1       62.849899          1.0   
43         sim_minus          8       19.010518          1.0   
42          sim_plus          8       17.433363          1.0   
47       total_minus          8        1.024075          1.0   
10        total_plus          2        0.666257          1.0   

    mean_interval_score  overall_rank  
9              4.832462           5.0  
2              9.480223          27.0  
43            46.212912          11.0  
42            49.241641          11.0  
47            56.553459          11.0  
10            23.608554           3.0  


In [11]:
metrics_to_plot = {
    "mean_rel_error",
    "pi_coverage",
    "mean_interval_score",
}

for model_type in results["model_type"].unique():

    kernel_results = results[
        results["model_type"] == model_type
    ].copy()

    fig, axes = performance_dist(kernel_results, metrics_to_plot, model_type)
    save_path = f"../../{folder}/plots/perform_dist_{model_type}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
